# Efficiency plotting 

Copied most functionality from Moon's https://github.com/wjdanswjddl/cafpyana/blob/release/numucc_1p0pi/analysis_village/numucc_1p0pi/notebooks/event_selection.ipynb

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# print avaialbe memory
import psutil
print(psutil.virtual_memory())

svmem(total=25769803776, available=11797086208, percent=54.2, used=10555228160, free=4429611008, active=7387283456, inactive=7051018240, wired=3167944704)


In [3]:
import os
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from os import makedirs, path
from datetime import datetime
import pickle

# local imports
# sys.path.append('../../../')
cwd = Path.cwd().resolve()
repo_root = next((candidate for candidate in [cwd, *cwd.parents] if (candidate / 'analysis_village').exists()), None)
if repo_root is None:
    raise RuntimeError('Could not locate the repository root from the current notebook working directory')
sys.path.append(str(repo_root))
from analysis_village.nueNp0Pi.config.plots import VariableConfig
from analysis_village.nueNp0Pi.config.settings import *
from analysis_village.nueNp0Pi.selections import *
from analysis_village.nueNp0Pi.utils import *
from pyanalib.split_df_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.covariance import *

import matplotlib.pyplot as plt 
from matplotlib.patches import Patch

plt.style.use(repo_root / 'analysis_village' / 'nueNp0Pi' / 'notebooks' / 'presentation.mplstyle')

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
# turn off RuntimeWarning
warnings.filterwarnings('ignore', category=RuntimeWarning)

os.environ['CAFPYANA_LOG_LEVEL'] = 'DEBUG'


In [4]:
from analysis_village.nueNp0Pi.event_selection import build_event_selection_pipeline
from pyanalib.event_selection_pipeline import EventSelectionPipelineConfig
from analysis_village.nueNp0Pi.config.datasets import PLOTS_BASE

today_str = datetime.now().strftime("%Y%m%d")
syst_tag = ""

# Optional overrides (None -> dated work dir under /exp/sbnd/data/users/$USER/...)
batch_work_base = f"/Users/micarrig/Desktop/SBND/cafpyana/dump/{today_str}"
batch_plots_dir = path.join(PLOTS_BASE, f"event_selection-{syst_tag}-{today_str}")

batch_cfg = EventSelectionPipelineConfig(
    work_base=batch_work_base,
    plots_dir=batch_plots_dir,
    max_job_bytes=int(1.0 * 1024**3),  # 1 GiB per job
    mc_univ_syst=(), #("Flux", "G4", "GENIE"),
    skip_existing_batches=False,
    aggregate_only=False,
    skip_aggregate=False,
    save_fig=True,
    show_fig=False,
    # max_files_per_sample=1,  # smoke test: one file per sample
)
pipeline = build_event_selection_pipeline(batch_cfg)

records, jobs, manifest_path = pipeline.discover_jobs()
print(f"Batched workflow: {len(jobs)} job(s)  manifest={manifest_path}")
for j in jobs[:8]:
    print(f"  {j.sample} {j.tag}: {len(j.files)} file(s), {j.total_bytes / (1024**3):.3f} GiB")
if len(jobs) > 8:
    print(f"  ... and {len(jobs) - 8} more")


[event_selection] mc: looking in /Users/micarrig/Desktop/SBND/data/*.df
[event_selection] mc: found 1 file(s)
[event_selection] sample=mc  files=1  total=6.81 GiB
[event_selection] 1 job(s) under size budget
  mc batch_0000: 1 file(s), 6.813 GiB
Batched workflow: 1 job(s)  manifest=/Users/micarrig/Desktop/SBND/cafpyana/dump/20260812/manifest.json
  mc batch_0000: 1 file(s), 6.813 GiB


### Stacked event-distribution breakdown plots

By default, `build_pipeline()` (in `config/stages.py`) attaches a stacked breakdown plot
for every `EFFICIENCY_VARS` variable, by topology, at the final selection stage -- no
changes needed below to get them. Set the `EVT_BREAKDOWN_*` attributes on the
`config.stages` module before running the pipeline (cell below) to change the
breakdown type, which stage(s) they're attached to, or which variables get one.

In [5]:
batch_result = pipeline.run_full()

save_fig_dir = str(batch_result.plots_dir)
save_fig = batch_cfg.save_fig
show_plot = batch_cfg.show_fig
pot_str = batch_result.pot_str
data_tot_pot = batch_result.data_pot
merged_payload = batch_result.merged_payload

print("Batched workflow complete.")
print("  batches:", batch_result.batches_dir)
print("  plots  :", batch_result.plots_dir)
print("  manifest:", batch_result.manifest_path)
print("  POT    :", pot_str)

# Uncomment to preview PNGs inline (can be slow for many plots):
# pipeline.show_saved_plots(batch_result.plots_dir, max_images=20)

[event_selection] mc: looking in /Users/micarrig/Desktop/SBND/data/*.df
[event_selection] mc: found 1 file(s)
[event_selection] sample=mc  files=1  total=6.81 GiB
[event_selection] 1 job(s) under size budget
  mc batch_0000: 1 file(s), 6.813 GiB
[event_selection] WORK_BASE=/Users/micarrig/Desktop/SBND/cafpyana/dump/20260812
[event_selection] BATCHES_DIR=/Users/micarrig/Desktop/SBND/cafpyana/dump/20260812/batches
[event_selection] sample=mc batch_0000  files=1  size=6.813 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[event_selection] done sample=mc batch_0000  n_evt=56335  pot=1.425e+19
[event_selection] sample=mc wrote 1 batch pickle(s)


[INFO] cafpyana.pyanalib.chunked_selection: aggregating 1 chunk file(s)
[INFO] cafpyana.pyanalib.chunked_selection: merging samples: ['mc']
[INFO] cafpyana.pyanalib.chunked_selection: applied global exposure scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}


[aggregate] sample=mc -> 1 chunks
[aggregate] sample=data -> 0 chunks
[aggregate] sample=intime -> 0 chunks
[aggregate] sample=offbeam -> 0 chunks
[aggregate] sample=dirt -> 0 chunks
[aggregate] aggregating 1 chunks for sample=mc
[aggregate] exposure totals: data_pot=0.000e+00 bnb_gates=0.000e+00 mc_pot=1.425e+19 dirt_pot=0.000e+00 intime_gates=0.000e+00 offbeam_gates=0.000e+00
[aggregate] applied global scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}
[aggregate] data_pot (legend)=1.425e+19 -> POT label=1.42$\times 10^{19}$
[aggregate] systematics disk root not found, skipping syst bands: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] overlay plots without syst covariance (10 plots, 10 distinct var_save_name): E_nu, electron-e, particle_ke, proton-p, tki-del_Tp, tki-del_Tp_lp, tki-del_alpha, tki-del_alpha_lp, tki-del_phi, tki-del_phi_lp
[aggregate] final selection purity: 87.64% (weighted)  87.64% (raw counts, notebook-style)
[aggregate] wrote /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260812/pkl/eff_dict.pkl
[aggregate] wrote /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260812/pkl/merged_histdata.pkl
[event_selection] DONE plots -> /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260812
Batched workflow complete.
  batches: /Users/micarrig/Desktop/SBND/cafpyana/dump/20260812/batches
  plots  : /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260812
  manifest: /Users/micarrig/Desktop/SBND/cafpyana/dump/20260812/manifest.json
  POT    : 1.42$\times 10^{19}$


In [6]:
import analysis_village.nueNp0Pi.config.stages as stages_mod

# Uncomment/edit any of these before running the pipeline below. Defaults shown.
stages_mod.EVT_BREAKDOWN_TYPE = "genie"           # "topology" | "genie" | "pdg"
stages_mod.EVT_BREAKDOWN_DIR_NAME = f"selection_{stages_mod.EVT_BREAKDOWN_TYPE}"
# stages_mod.EVT_BREAKDOWN_STAGE_KEYS = None            # None -> final stage only; or e.g. ["no_photons", "electron_dedx"]
# stages_mod.EVT_BREAKDOWN_VARS = stages_mod.EFFICIENCY_VARS  # or e.g. [VariableConfig.neutrino_energy()]
# stages_mod.EVT_BREAKDOWN_VARS = []                    # uncomment to disable these plots entirely

# Output subdirectory for these plots (under the plots dir), default "selection". Set this
# alongside EVT_BREAKDOWN_TYPE if you want to run more than one breakdown_type (e.g. once as
# "topology", once as "genie") without the later run's selection_<var>.png files
# overwriting the earlier run's -- e.g.:
# stages_mod.EVT_BREAKDOWN_DIR_NAME = f"selection_{stages_mod.EVT_BREAKDOWN_TYPE}"


In [7]:
batch_result = pipeline.run_full()

save_fig_dir = str(batch_result.plots_dir)
save_fig = batch_cfg.save_fig
show_plot = batch_cfg.show_fig
pot_str = batch_result.pot_str
data_tot_pot = batch_result.data_pot
merged_payload = batch_result.merged_payload

print("Batched workflow complete.")
print("  batches:", batch_result.batches_dir)
print("  plots  :", batch_result.plots_dir)
print("  manifest:", batch_result.manifest_path)
print("  POT    :", pot_str)

# Uncomment to preview PNGs inline (can be slow for many plots):
# pipeline.show_saved_plots(batch_result.plots_dir, max_images=20)


[event_selection] mc: looking in /Users/micarrig/Desktop/SBND/data/*.df
[event_selection] mc: found 1 file(s)
[event_selection] sample=mc  files=1  total=6.81 GiB
[event_selection] 1 job(s) under size budget
  mc batch_0000: 1 file(s), 6.813 GiB
[event_selection] WORK_BASE=/Users/micarrig/Desktop/SBND/cafpyana/dump/20260812
[event_selection] BATCHES_DIR=/Users/micarrig/Desktop/SBND/cafpyana/dump/20260812/batches
[event_selection] sample=mc batch_0000  files=1  size=6.813 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[event_selection] done sample=mc batch_0000  n_evt=56335  pot=1.425e+19
[event_selection] sample=mc wrote 1 batch pickle(s)


[INFO] cafpyana.pyanalib.chunked_selection: aggregating 1 chunk file(s)
[INFO] cafpyana.pyanalib.chunked_selection: merging samples: ['mc']
[INFO] cafpyana.pyanalib.chunked_selection: applied global exposure scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}


[aggregate] sample=mc -> 1 chunks
[aggregate] sample=data -> 0 chunks
[aggregate] sample=intime -> 0 chunks
[aggregate] sample=offbeam -> 0 chunks
[aggregate] sample=dirt -> 0 chunks
[aggregate] aggregating 1 chunks for sample=mc
[aggregate] exposure totals: data_pot=0.000e+00 bnb_gates=0.000e+00 mc_pot=1.425e+19 dirt_pot=0.000e+00 intime_gates=0.000e+00 offbeam_gates=0.000e+00
[aggregate] applied global scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}
[aggregate] data_pot (legend)=1.425e+19 -> POT label=1.42$\times 10^{19}$
[aggregate] systematics disk root not found, skipping syst bands: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] overlay plots without syst covariance (10 plots, 10 distinct var_save_name): E_nu, electron-e, particle_ke, proton-p, tki-del_Tp, tki-del_Tp_lp, tki-del_alpha, tki-del_alpha_lp, tki-del_phi, tki-del_phi_lp
[aggregate] final selection purity: 87.64% (weighted)  87.64% (raw counts, notebook-style)
[aggregate] wrote /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260812/pkl/eff_dict.pkl
[aggregate] wrote /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260812/pkl/merged_histdata.pkl
[event_selection] DONE plots -> /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260812
Batched workflow complete.
  batches: /Users/micarrig/Desktop/SBND/cafpyana/dump/20260812/batches
  plots  : /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260812
  manifest: /Users/micarrig/Desktop/SBND/cafpyana/dump/20260812/manifest.json
  POT    : 1.42$\times 10^{19}$
